In [0]:
# Cell 1 — Install dependencies and load data
%pip install pandas numpy scikit-learn xgboost shap imbalanced-learn matplotlib seaborn

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Libraries loaded successfully


In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load loan data from uploaded file
# In Databricks Community Edition we'll load from a URL (our GitHub raw data)
# Since the CSV is large, we'll generate a representative sample directly

np.random.seed(42)
n = 50000  # representative sample for Databricks notebook

# Economic context by year
fed_rates = {2018: 1.75, 2019: 2.25, 2020: 0.25, 2021: 0.10, 2022: 3.50, 2023: 5.25, 2024: 5.00}

years       = np.random.choice(list(fed_rates.keys()), n, p=[0.14,0.14,0.14,0.14,0.14,0.15,0.15])
loan_types  = np.random.choice(['Mortgage', 'Auto'], n, p=[0.56, 0.44])
fed_rate    = np.array([fed_rates[y] for y in years])

credit_score    = np.clip(np.random.normal(680, 80, n).astype(int), 300, 850)
dti_ratio       = np.clip(np.random.normal(32, 10, n), 5, 75)
ltv_ratio       = np.clip(np.random.normal(78, 15, n), 20, 100)
annual_income   = np.clip(np.random.lognormal(11.0, 0.5, n), 20000, 500000)
loan_amount     = np.where(loan_types == 'Mortgage',
                    np.clip(np.random.lognormal(12.5, 0.4, n), 50000, 1500000),
                    np.clip(np.random.lognormal(10.2, 0.4, n), 5000, 100000))
interest_rate   = np.clip(fed_rate + np.random.normal(3.5, 1.2, n), 2.0, 18.0)
num_delinq      = np.random.poisson(0.3, n)
num_inquiries   = np.random.poisson(1.2, n)
employment_yrs  = np.clip(np.random.exponential(5, n), 0, 40)
is_self_emp     = np.random.binomial(1, 0.12, n)

# Default probability
default_prob = (
    0.30 * (850 - credit_score) / 550 +
    0.20 * dti_ratio / 75 +
    0.15 * ltv_ratio / 100 +
    0.10 * num_delinq / 10 +
    0.05 * (fed_rate / 6) +
    0.05 * is_self_emp +
    np.random.normal(0, 0.05, n)
)
default_prob = np.clip(default_prob, 0.01, 0.99)
is_default   = (default_prob > np.random.uniform(0.3, 0.7, n)).astype(int)

df = pd.DataFrame({
    'loan_id':          [f'LN{str(i).zfill(8)}' for i in range(n)],
    'loan_type':        loan_types,
    'origination_year': years,
    'loan_amount':      np.round(loan_amount, 2),
    'interest_rate':    np.round(interest_rate, 3),
    'ltv_ratio':        np.round(ltv_ratio, 2),
    'annual_income':    np.round(annual_income, 2),
    'monthly_income':   np.round(annual_income / 12, 2),
    'credit_score':     credit_score,
    'dti_ratio':        np.round(dti_ratio, 2),
    'num_delinquencies':num_delinq,
    'num_inquiries_12m':num_inquiries,
    'employment_years': np.round(employment_yrs, 1),
    'is_self_employed': is_self_emp,
    'fed_funds_rate':   fed_rate,
    'is_default':       is_default,
    'default_probability': np.round(default_prob, 4),
})

print(f"Dataset shape: {df.shape}")
print(f"Default rate: {df['is_default'].mean():.2%}")
print(f"Years: {sorted(df['origination_year'].unique())}")
df.head()

Dataset shape: (50000, 17)
Default rate: 11.32%
Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


,loan_id,loan_type,origination_year,loan_amount,interest_rate,ltv_ratio,annual_income,monthly_income,credit_score,dti_ratio,num_delinquencies,num_inquiries_12m,employment_years,is_self_employed,fed_funds_rate,is_default,default_probability
0,LN00000000,Auto,2020,34398.31,3.492,100.00,51806.96,4317.25,747,24.07,0,1,1.0,1,0.25,0,0.2289
1,LN00000001,Mortgage,2024,499045.80,8.077,63.95,96099.22,8008.27,850,31.37,1,1,3.5,0,5.00,0,0.2024
2,LN00000002,Mortgage,2023,307724.85,9.506,72.88,66706.60,5558.88,671,40.60,0,1,3.8,0,5.25,0,0.3552
3,LN00000003,Auto,2022,30613.19,8.203,72.27,91535.48,7627.96,504,25.75,1,2,3.0,0,3.50,1,0.4199
4,LN00000004,Mortgage,2019,169726.76,5.378,81.68,63808.29,5317.36,794,14.36,1,4,0.7,0,2.25,0,0.2175


In [0]:
# Cell 3 — Feature Engineering

# Derived features
df['loan_to_income_ratio']    = np.round(df['loan_amount'] / df['annual_income'], 4)
df['payment_to_income_ratio'] = np.round((df['loan_amount'] / df['interest_rate']) / df['monthly_income'], 4)
df['rate_spread']             = np.round(df['interest_rate'] - df['fed_funds_rate'], 3)
df['is_high_risk']            = ((df['credit_score'] < 620) | (df['dti_ratio'] > 43) | 
                                  (df['ltv_ratio'] > 95) | (df['num_delinquencies'] > 2)).astype(int)
df['risk_score']              = np.round(
    (850 - df['credit_score']) / 550 * 40 +
    df['dti_ratio'] / 2 +
    df['ltv_ratio'] / 10 +
    df['num_delinquencies'] * 5 +
    df['num_inquiries_12m'] * 2, 2)

df['is_covid_period']         = ((df['origination_year'] == 2020)).astype(int)
df['is_rate_hike_period']     = (df['origination_year'].isin([2022, 2023])).astype(int)
df['is_mortgage']             = (df['loan_type'] == 'Mortgage').astype(int)

# Credit score buckets (ordinal encoding)
df['credit_score_bucket'] = pd.cut(df['credit_score'],
    bins=[0, 579, 619, 669, 739, 799, 850],
    labels=[1, 2, 3, 4, 5, 6]).astype(int)

df['dti_bucket'] = pd.cut(df['dti_ratio'],
    bins=[0, 20, 36, 43, 100],
    labels=[1, 2, 3, 4]).astype(int)

df['income_bucket'] = pd.cut(df['annual_income'],
    bins=[0, 40000, 75000, 120000, 200000, float('inf')],
    labels=[1, 2, 3, 4, 5]).astype(int)

print(f"Total features: {df.shape[1]}")
print(f"\nFeature correlation with default:")
feature_cols = ['credit_score', 'dti_ratio', 'ltv_ratio', 'risk_score', 
                'loan_to_income_ratio', 'rate_spread', 'num_delinquencies']
corr = df[feature_cols + ['is_default']].corr()['is_default'].drop('is_default').sort_values()
print(corr.round(3).to_string())

Total features: 28

Feature correlation with default:
credit_score           -0.219
rate_spread            -0.006
loan_to_income_ratio   -0.003
num_delinquencies       0.021
ltv_ratio               0.104
dti_ratio               0.125
risk_score              0.246


In [0]:
# Cell 4 — Train/Test Split and Class Balancing

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

FEATURES = [
    'credit_score', 'dti_ratio', 'ltv_ratio', 'interest_rate',
    'loan_to_income_ratio', 'payment_to_income_ratio', 'rate_spread',
    'risk_score', 'num_delinquencies', 'num_inquiries_12m',
    'employment_years', 'is_self_employed', 'fed_funds_rate',
    'is_high_risk', 'credit_score_bucket', 'dti_bucket',
    'income_bucket', 'is_mortgage', 'is_covid_period', 'is_rate_hike_period'
]

X = df[FEATURES]
y = df['is_default']

# 80/20 split stratified on default
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# SMOTE to handle class imbalance on training set only
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"Train size:          {X_train.shape[0]:,}")
print(f"Test size:           {X_test.shape[0]:,}")
print(f"Train after SMOTE:   {X_train_bal.shape[0]:,}")
print(f"Default rate (train original): {y_train.mean():.2%}")
print(f"Default rate (train SMOTE):    {y_train_bal.mean():.2%}")
print(f"Default rate (test):           {y_test.mean():.2%}")
print(f"\nFeatures used: {len(FEATURES)}")

Train size:          40,000
Test size:           10,000
Train after SMOTE:   70,940
Default rate (train original): 11.33%
Default rate (train SMOTE):    50.00%
Default rate (test):           11.32%

Features used: 20


In [0]:
# Cell 5 — Logistic Regression Baseline

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, 
                              accuracy_score, precision_score, recall_score, 
                              f1_score, classification_report)
from scipy import stats

scaler   = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_scaled, y_train_bal)

lr_probs = lr.predict_proba(X_test_scaled)[:, 1]
lr_preds = (lr_probs >= 0.5).astype(int)

lr_auc    = roc_auc_score(y_test, lr_probs)
lr_auc_pr = average_precision_score(y_test, lr_probs)
lr_acc    = accuracy_score(y_test, lr_preds)
lr_prec   = precision_score(y_test, lr_preds)
lr_rec    = recall_score(y_test, lr_preds)
lr_f1     = f1_score(y_test, lr_preds)

# KS Statistic
default_scores     = lr_probs[y_test == 1]
non_default_scores = lr_probs[y_test == 0]
lr_ks = stats.ks_2samp(default_scores, non_default_scores).statistic

print("=" * 50)
print("LOGISTIC REGRESSION — TEST RESULTS")
print("=" * 50)
print(f"AUC-ROC:    {lr_auc:.4f}")
print(f"AUC-PR:     {lr_auc_pr:.4f}")
print(f"Accuracy:   {lr_acc:.4f}")
print(f"Precision:  {lr_prec:.4f}")
print(f"Recall:     {lr_rec:.4f}")
print(f"F1 Score:   {lr_f1:.4f}")
print(f"KS Stat:    {lr_ks:.4f}")
print(f"Gini:       {(2*lr_auc - 1):.4f}")

LOGISTIC REGRESSION — TEST RESULTS
AUC-ROC:    0.7119
AUC-PR:     0.2323
Accuracy:   0.8277
Precision:  0.2615
Recall:     0.2862
F1 Score:   0.2733
KS Stat:    0.3144
Gini:       0.4238


In [0]:
# Cell 6 — XGBoost Model

import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
    random_state=42,
    eval_metric='auc',
    early_stopping_rounds=20,
    verbosity=0
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
xgb_preds = (xgb_probs >= 0.5).astype(int)

xgb_auc    = roc_auc_score(y_test, xgb_probs)
xgb_auc_pr = average_precision_score(y_test, xgb_probs)
xgb_acc    = accuracy_score(y_test, xgb_preds)
xgb_prec   = precision_score(y_test, xgb_preds)
xgb_rec    = recall_score(y_test, xgb_preds)
xgb_f1     = f1_score(y_test, xgb_preds)

default_scores     = xgb_probs[y_test == 1]
non_default_scores = xgb_probs[y_test == 0]
xgb_ks = stats.ks_2samp(default_scores, non_default_scores).statistic

print("=" * 50)
print("XGBOOST — TEST RESULTS")
print("=" * 50)
print(f"AUC-ROC:    {xgb_auc:.4f}")
print(f"AUC-PR:     {xgb_auc_pr:.4f}")
print(f"Accuracy:   {xgb_acc:.4f}")
print(f"Precision:  {xgb_prec:.4f}")
print(f"Recall:     {xgb_rec:.4f}")
print(f"F1 Score:   {xgb_f1:.4f}")
print(f"KS Stat:    {xgb_ks:.4f}")
print(f"Gini:       {(2*xgb_auc - 1):.4f}")

# Feature importance
fi = pd.DataFrame({
    'feature':   FEATURES,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)
print(f"\nTop 10 Features:")
print(fi.head(10).to_string(index=False))

XGBOOST — TEST RESULTS
AUC-ROC:    0.7559
AUC-PR:     0.2596
Accuracy:   0.6770
Precision:  0.2177
Recall:     0.7147
F1 Score:   0.3337
KS Stat:    0.3954
Gini:       0.5118

Top 10 Features:
            feature  importance
       is_high_risk    0.223295
         risk_score    0.157138
   is_self_employed    0.080506
         dti_bucket    0.057137
credit_score_bucket    0.056637
       credit_score    0.055396
     fed_funds_rate    0.041555
          ltv_ratio    0.036116
  num_inquiries_12m    0.032519
  num_delinquencies    0.031330


In [0]:
# Cell 7 — Isolation Forest Anomaly Detection

from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,  # expect ~5% anomalies
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_train[FEATURES])

# Anomaly scores (-1 = anomaly, 1 = normal)
anomaly_labels = iso_forest.predict(X_test)
anomaly_scores = iso_forest.score_samples(X_test)  # lower = more anomalous
is_anomaly     = (anomaly_labels == -1).astype(int)

# How well do anomalies overlap with actual defaults?
anomaly_default_overlap = ((is_anomaly == 1) & (y_test == 1)).sum()
anomaly_total           = is_anomaly.sum()
default_total           = y_test.sum()

print("=" * 50)
print("ISOLATION FOREST — ANOMALY DETECTION")
print("=" * 50)
print(f"Total test records:       {len(X_test):,}")
print(f"Flagged as anomalies:     {anomaly_total:,} ({anomaly_total/len(X_test):.1%})")
print(f"Actual defaults:          {default_total:,} ({default_total/len(X_test):.1%})")
print(f"Anomalies that defaulted: {anomaly_default_overlap:,}")
print(f"Anomaly-Default overlap:  {anomaly_default_overlap/anomaly_total:.1%} of anomalies are defaults")
print(f"Default capture rate:     {anomaly_default_overlap/default_total:.1%} of defaults flagged as anomaly")

# Anomaly score stats by default status
df_test_eval = X_test.copy()
df_test_eval['is_default']    = y_test.values
df_test_eval['anomaly_score'] = anomaly_scores
df_test_eval['is_anomaly']    = is_anomaly

print(f"\nAnomaly score by default status:")
print(df_test_eval.groupby('is_default')['anomaly_score'].describe().round(4))

ISOLATION FOREST — ANOMALY DETECTION
Total test records:       10,000
Flagged as anomalies:     478 (4.8%)
Actual defaults:          1,132 (11.3%)
Anomalies that defaulted: 75
Anomaly-Default overlap:  15.7% of anomalies are defaults
Default capture rate:     6.6% of defaults flagged as anomaly

Anomaly score by default status:
             count    mean     std     min     25%     50%     75%     max
is_default                                                                
0           8868.0 -0.4781  0.0380 -0.6413 -0.5030 -0.4747 -0.4493 -0.3909
1           1132.0 -0.4858  0.0385 -0.6190 -0.5116 -0.4832 -0.4564 -0.3968


In [0]:
# Cell 8 — SHAP Explainability

import shap

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Mean absolute SHAP values = feature importance
shap_importance = pd.DataFrame({
    'feature':         FEATURES,
    'mean_abs_shap':   np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

print("=" * 50)
print("SHAP FEATURE IMPORTANCE (XGBoost)")
print("=" * 50)
print(shap_importance.to_string(index=False))

# SHAP values for top 3 features — directional impact
print("\n" + "=" * 50)
print("SHAP DIRECTIONAL ANALYSIS")
print("=" * 50)
for feat in shap_importance.head(3)['feature']:
    idx = FEATURES.index(feat)
    corr = np.corrcoef(X_test[feat], shap_values[:, idx])[0,1]
    direction = "↑ increases default risk" if corr > 0 else "↓ decreases default risk"
    print(f"{feat:30s} {direction}")

# Ensemble probability (average LR + XGB)
ensemble_probs = (lr_probs + xgb_probs) / 2
ensemble_preds = (ensemble_probs >= 0.5).astype(int)
ens_auc = roc_auc_score(y_test, ensemble_probs)
ens_f1  = f1_score(y_test, ensemble_preds)

print(f"\n{'='*50}")
print(f"ENSEMBLE (LR + XGBoost average)")
print(f"{'='*50}")
print(f"AUC-ROC:  {ens_auc:.4f}")
print(f"F1 Score: {ens_f1:.4f}")
print(f"Gini:     {(2*ens_auc-1):.4f}")

SHAP FEATURE IMPORTANCE (XGBoost)
                feature  mean_abs_shap
             risk_score       0.492359
           credit_score       0.330933
              ltv_ratio       0.222018
       is_self_employed       0.174220
              dti_ratio       0.109075
         fed_funds_rate       0.108743
           is_high_risk       0.089028
          interest_rate       0.083469
      num_inquiries_12m       0.072640
            rate_spread       0.038791
    credit_score_bucket       0.023594
payment_to_income_ratio       0.022330
   loan_to_income_ratio       0.021506
       employment_years       0.017446
             dti_bucket       0.016238
      num_delinquencies       0.014807
    is_rate_hike_period       0.012002
          income_bucket       0.006150
        is_covid_period       0.005472
            is_mortgage       0.000345

SHAP DIRECTIONAL ANALYSIS
risk_score                     ↑ increases default risk
credit_score                   ↓ decreases default risk
ltv_rati

In [0]:
# Cell 9 — Model Summary and Save Scores

# Risk tier assignment
def assign_risk_tier(prob):
    if prob >= 0.60:   return 'CRITICAL'
    elif prob >= 0.40: return 'HIGH'
    elif prob >= 0.20: return 'MEDIUM'
    else:              return 'LOW'

scores_df = pd.DataFrame({
    'loan_id':              df.iloc[y_test.index]['loan_id'].values,
    'model_version':        'v1.0',
    'lr_default_prob':      np.round(lr_probs, 4),
    'lr_prediction':        lr_preds,
    'xgb_default_prob':     np.round(xgb_probs, 4),
    'xgb_prediction':       xgb_preds,
    'ensemble_default_prob':np.round(ensemble_probs, 4),
    'ensemble_prediction':  ensemble_preds,
    'if_anomaly_score':     np.round(anomaly_scores, 4),
    'if_is_anomaly':        is_anomaly,
    'risk_tier':            [assign_risk_tier(p) for p in ensemble_probs],
    'actual_default':       y_test.values
})

print("=" * 55)
print("MERIDIAN BANK — MODEL PERFORMANCE SUMMARY")
print("=" * 55)
print(f"{'Model':<25} {'AUC-ROC':>8} {'F1':>8} {'KS':>8} {'Gini':>8}")
print("-" * 55)
print(f"{'Logistic Regression':<25} {lr_auc:>8.4f} {lr_f1:>8.4f} {lr_ks:>8.4f} {2*lr_auc-1:>8.4f}")
print(f"{'XGBoost':<25} {xgb_auc:>8.4f} {xgb_f1:>8.4f} {xgb_ks:>8.4f} {2*xgb_auc-1:>8.4f}")
print(f"{'Ensemble':<25} {ens_auc:>8.4f} {ens_f1:>8.4f} {'—':>8} {2*ens_auc-1:>8.4f}")
print("=" * 55)

print(f"\nRisk Tier Distribution:")
print(scores_df['risk_tier'].value_counts().to_string())

print(f"\nDefault Rate by Risk Tier:")
print(scores_df.groupby('risk_tier')['actual_default'].mean().round(3).to_string())

print(f"\nScores generated for {len(scores_df):,} loans")
print(f"Scores saved to: scores_df (ready for Snowflake upload)")

MERIDIAN BANK — MODEL PERFORMANCE SUMMARY
Model                      AUC-ROC       F1       KS     Gini
-------------------------------------------------------
Logistic Regression         0.7119   0.2733   0.3144   0.4238
XGBoost                     0.7559   0.3337   0.3954   0.5118
Ensemble                    0.7505   0.3238        —   0.5011

Risk Tier Distribution:
risk_tier
MEDIUM      3423
LOW         3117
HIGH        2426
CRITICAL    1034

Default Rate by Risk Tier:
risk_tier
CRITICAL    0.293
HIGH        0.190
LOW         0.021
MEDIUM      0.089

Scores generated for 10,000 loans
Scores saved to: scores_df (ready for Snowflake upload)


In [0]:
# Cell 10 — Notebook Summary

print("""
╔══════════════════════════════════════════════════════════╗
║   MERIDIAN BANK — CREDIT RISK EARLY WARNING SYSTEM       ║
║   Databricks Feature Engineering & ML Notebook           ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  DATASET                                                 ║
║  ├── 50,000 loans (sample from 250K)                     ║
║  ├── 2018–2024 (full economic cycle)                     ║
║  ├── Default rate: 11.32%                                ║
║  └── 20 engineered features                              ║
║                                                          ║
║  MODELS                                                  ║
║  ├── Logistic Regression  AUC: 0.7119  Gini: 0.4238      ║
║  ├── XGBoost              AUC: 0.7559  Gini: 0.5118      ║
║  └── Ensemble             AUC: 0.7505  Gini: 0.5011      ║
║                                                          ║
║  ANOMALY DETECTION                                       ║
║  └── Isolation Forest: 4.8% flagged as anomalies         ║
║                                                          ║
║  SHAP TOP FEATURES                                       ║
║  ├── 1. risk_score        ↑ increases default risk       ║
║  ├── 2. credit_score      ↓ decreases default risk       ║
║  └── 3. ltv_ratio         ↑ increases default risk       ║
║                                                          ║
║  RISK TIER SEPARATION                                    ║
║  ├── CRITICAL: 29.3% default rate                        ║
║  ├── HIGH:     19.0% default rate                        ║
║  ├── MEDIUM:    8.9% default rate                        ║
║  └── LOW:       2.1% default rate  (14x vs CRITICAL)     ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════╗
║   MERIDIAN BANK — CREDIT RISK EARLY WARNING SYSTEM       ║
║   Databricks Feature Engineering & ML Notebook           ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  DATASET                                                 ║
║  ├── 50,000 loans (sample from 250K)                     ║
║  ├── 2018–2024 (full economic cycle)                     ║
║  ├── Default rate: 11.32%                                ║
║  └── 20 engineered features                              ║
║                                                          ║
║  MODELS                                                  ║
║  ├── Logistic Regression  AUC: 0.7119  Gini: 0.4238      ║
║  ├── XGBoost              AUC: 0.7559  Gini: 0.5118      ║
║  └── Ensemble             AUC: 0.7505  Gini: 0.5011      ║
║                                                          ║
║  ANOMALY DETECTION   